# ETL para Dataset de Match entre Candidatos e Vagas

Este notebook realiza a extração, transformação e consolidação dos dados dos arquivos `vagas.json`, `applicants.json` e `prospects.json`, com objetivo de gerar um dataset estruturado para treinamento de um modelo de machine learning que prediz o *match* entre candidatos e vagas.

In [1]:
import os
import json
import pandas as pd
import re

## Funções auxiliares
Funções para padronização de níveis, limpeza de texto e verificação de aprovação.

In [2]:
def padronizar_nivel(texto, mapa_niveis):
    if not texto or not isinstance(texto, str):
        return "Desconhecido"
    texto = texto.strip().lower()
    for chave, valor in mapa_niveis.items():
        if chave in texto:
            return valor
    return "Outro"

def limpar_texto(texto):
    if not texto or not isinstance(texto, str):
        return ""
    return re.sub(r"\s+", " ", texto).strip()

def verificar_aprovacao(situacao):
    if not situacao or not isinstance(situacao, str):
        return 0
    situacao = situacao.strip().lower()
    situacoes_positivas = {
        "aprovado",
        "proposta aceita",
        "contratado como hunting",
        "contratado pela decision"
    }
    return 1 if situacao in situacoes_positivas else 0

## Mapas de padronização
Definições para conversão dos níveis em valores padronizados.

In [3]:
mapa_nivel_ingles = {
    "básico": "Básico", "intermediário": "Intermediário",
    "avançado": "Avançado", "fluente": "Fluente",
    "não informado": "Desconhecido", "desconhecido": "Desconhecido"
}
mapa_nivel_espanhol = mapa_nivel_ingles
mapa_nivel_academico = {
    "fundamental": "Fundamental", "médio": "Médio", "superior": "Superior",
    "pós": "Pós-graduação", "mestrado": "Mestrado", "doutorado": "Doutorado",
    "não informado": "Desconhecido", "desconhecido": "Desconhecido"
}

## Carregamento dos dados
Leitura dos arquivos JSON de vagas, candidatos e interações (prospects).

In [4]:
with open("../data/vagas.json", encoding="utf-8") as f:
    jobs_data = json.load(f)

with open("../data/applicants.json", encoding="utf-8") as f:
    applicants_data = json.load(f)

with open("../data/prospects.json", encoding="utf-8") as f:
    prospects_data = json.load(f)

## Processamento dos registros
Loop principal que une os dados de vagas e candidatos a partir dos prospects.

In [ ]:
registros = []

for vaga_id, prospect_info in prospects_data.items():
    vaga = jobs_data.get(vaga_id)
    if not vaga:
        continue

    perfil = vaga.get("perfil_vaga", {})
    nivel_academico_vaga = padronizar_nivel(perfil.get("nivel_academico"), mapa_nivel_academico)
    tecnicos = perfil.get("competencia_tecnicas_e_comportamentais", "")
    atividades = perfil.get("principais_atividades", "")
    requisitos = limpar_texto(f"{tecnicos} {atividades}")

    for prospect in prospect_info.get("prospects", []):
        codigo_candidato = prospect.get("codigo")
        candidato = applicants_data.get(codigo_candidato, {})

        info_formacao = candidato.get("formacao_e_idiomas", {})
        info_profissional = candidato.get("informacoes_profissionais", {})
        cv_raw = candidato.get("cv_pt", "")

        nivel_ingles_vaga = padronizar_nivel(perfil.get("nivel_ingles"), mapa_nivel_ingles)
        nivel_espanhol_vaga = padronizar_nivel(perfil.get("nivel_espanhol"), mapa_nivel_espanhol)
        nivel_academico = padronizar_nivel(info_formacao.get("nivel_academico"), mapa_nivel_academico)
        nivel_ingles = padronizar_nivel(info_formacao.get("nivel_ingles"), mapa_nivel_ingles)
        nivel_espanhol = padronizar_nivel(info_formacao.get("nivel_espanhol"), mapa_nivel_espanhol)

        cv_texto = limpar_texto(cv_raw)
        conhecimentos_tecnicos = limpar_texto(info_profissional.get("conhecimentos_tecnicos", ""))

        titulo_vaga = vaga.get("informacoes_basicas", {}).get("titulo_vaga", "Desconhecido")
        cliente = vaga.get("informacoes_basicas", {}).get("cliente", "Desconhecido")
        nivel_vaga = perfil.get("nivel profissional") or "Desconhecido"
        local_vaga = perfil.get("cidade") or "Desconhecido"

        nome_candidato = candidato.get("informacoes_pessoais", {}).get("nome", "Desconhecido")
        situacao = prospect.get("situacao_candidado", "Não informado")
        comentario = prospect.get("comentario", "")
        recrutador = prospect.get("recrutador", "")

        registros.append({
            "vaga_id": vaga_id,
            "titulo_vaga": titulo_vaga,
            "cliente": cliente,
            "nivel_profissional_vaga": nivel_vaga,
            "nivel_academico_vaga": nivel_academico_vaga,
            "nivel_ingles_vaga": nivel_ingles_vaga,
            "nivel_espanhol_vaga": nivel_espanhol_vaga,
            "local_vaga": local_vaga,
            "requisitos_vaga": requisitos,
            "codigo_candidato": codigo_candidato if codigo_candidato else "Desconhecido",
            "nome_candidato": nome_candidato,
            "nivel_academico_candidato": nivel_academico,
            "nivel_ingles_candidato": nivel_ingles,
            "nivel_espanhol_candidato": nivel_espanhol,
            "conhecimentos_tecnicos": conhecimentos_tecnicos,
            "cv_texto": cv_texto,
            "nivel_profissional_candidato": info_profissional.get("nivel_profissional", "Desconhecido"),
            "local_candidato": info_profissional.get("cidade", "Desconhecido"),
            "situacao": situacao,
            "comentario": comentario,
            "recrutador": recrutador
        })

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' on line 41 (4264510919.py, line 63)

## Criação do DataFrame
Consolida os registros em um DataFrame e gera a coluna `match`.

In [ ]:
df = pd.DataFrame(registros)
df["match"] = df["situacao"].apply(verificar_aprovacao)

colunas = [col for col in df.columns if col != "match"]
df = df[colunas + ["match"]]

## Análise da distribuição original

In [ ]:
total_registros = len(df)
contagem_match = df["match"].value_counts()
aprovados = contagem_match.get(1, 0)
nao_aprovados = contagem_match.get(0, 0)

print(f"Total de registros: {total_registros}")
print(f"Aprovados: {aprovados} ({aprovados / total_registros * 100:.2f}%)")
print(f"Não aprovados: {nao_aprovados} ({nao_aprovados / total_registros * 100:.2f}%)")

## Balanceamento da base (1:1)

In [ ]:
positivos = df[df["match"] == 1]
negativos = df[df["match"] == 0]
n_negativos_desejado = len(positivos)

negativos_amostrado = negativos.sample(n=min(n_negativos_desejado, len(negativos)), random_state=42)

df_balanceado = pd.concat([positivos, negativos_amostrado]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribuição balanceada:")
print(df_balanceado["match"].value_counts(normalize=True).round(2))

## Salvar os arquivos de saída

In [ ]:
output_path = os.path.abspath("../output")
os.makedirs(output_path, exist_ok=True)

df.to_csv(os.path.join(output_path, "dataset_unificado.csv"), index=False, encoding="utf-8-sig")
df_balanceado.to_csv(os.path.join(output_path, "dataset_unificado_balanceado.csv"), index=False, encoding="utf-8-sig")

print(f"Original: {len(df)} registros | Balanceado: {len(df_balanceado)} registros")